In [12]:
# 1. System Requirements & Installations
!apt-get update && apt-get install zstd -y
!curl -L https://ollama.com/install.sh | sh
!pip install ollama

# 2. Wake up the Brain
import subprocess
import time

print("Starting the Ollama server...")
# We use 'Popen' to keep the server running in the background while you use the wallet
subprocess.Popen(["ollama", "serve"])

# Give the server 10 seconds to fully initialize
time.sleep(10)

# 3. Download the Gemma model
!ollama pull gemma2:2b
print("\n--- Setup Complete: The Brain is Ready! ---")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,917 B in 3s (1,257 B/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3bu

In [14]:
import ollama
try:
    # We simplified it to just check if it can talk to the brain
    response = ollama.list()
    print("HANDSHAKE SUCCESSFUL!")
    print("Your ThinkBook is now connected to the Brain.")

    # This part was causing the error, let's do it safely:
    for model in response.get('models', []):
        print(f"Model ready: {model.get('model', 'Unknown')}")

except Exception as e:
    print(f"Handshake failed: {e}")

HANDSHAKE SUCCESSFUL!
Your ThinkBook is now connected to the Brain.
Model ready: gemma2:2b


In [22]:
import subprocess
import time
import ollama

# 1. Kill any 'ghost' processes that are stuck
!pkill ollama

# 2. Restart the brain
print("Re-waking the advisor...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

# 3. Wait for it to wake up
time.sleep(10)

# 4. Quick Handshake Check
try:
    ollama.list()
    print("✅ HANDSHAKE SUCCESSFUL! Your advisor is back in the palace.")
except:
    print("❌ Still sleeping. Run this cell one more time.")

Re-waking the advisor...
✅ HANDSHAKE SUCCESSFUL! Your advisor is back in the palace.


In [ ]:
import ollama

def consult_ai_advisor(balances):
    print("\n--- Consulting AI Financial Advisor ---")

    # This turns your balances into a sentence the AI can understand
    wallet_summary = ", ".join([f"{curr}: {amt}" for curr, amt in balances.items()])

    print(f"Portfolio Data Sent to AI: {wallet_summary}")
    user_query = input("\nWhat is your financial question? (or type 'exit'): ")

    if user_query.lower() == 'exit':
        return

    try:
        # Calling the brain!
        response = ollama.chat(model='gemma2:2b', messages=[
            {
                'role': 'system',
                'content': f"You are a professional FinTech advisor. The user's balances are: {wallet_summary}. Give very short, expert advice."
            },
            {'role': 'user', 'content': user_query},
        ])
        print(f"\nADVISOR RESPONSE: {response['message']['content']}")
    except Exception as e:
        print(f"Error: The advisor is busy. ({e})")

In [ ]:
import csv
import os
import pandas as pd
import matplotlib.pyplot as plt


WALLET_FILE = 'wallet_balances.csv'


EXCHANGE_RATES = {
    'USD': 1.00,
    'EUR': 0.92,
    'GBP': 0.79,
    'JPY': 150.50
}

import ollama

def consult_ai_advisor(balances):
    print("\n--- Consulting AI Financial Advisor ---")
    wallet_summary = ", ".join([f"{curr}: {amt}" for curr, amt in balances.items()])

    user_query = input("\nWhat is your financial question? (or type 'exit'): ")
    if user_query.lower() == 'exit': return

    try:
        print("\nAdvisor is thinking...", end="", flush=True)

        # We add 'stream=True' so we see the words as they are born
        stream = ollama.chat(
            model='gemma2:2b',
            messages=[
                {'role': 'system', 'content': f"You are a FinTech advisor. User has: {wallet_summary}."},
                {'role': 'user', 'content': user_query},
            ],
            stream=True,
        )

        print("\nADVISOR RESPONSE: ", end="")
        for chunk in stream:
            print(chunk['message']['content'], end="", flush=True)
        print("\n") # New line at the end

    except Exception as e:
        print(f"\nError: {e}")

def initialize_wallet():

    if not os.path.exists(WALLET_FILE):
        default_balances = {'USD': 0.0, 'EUR': 0.0, 'GBP': 0.0, 'JPY': 0.0}
        save_balances(default_balances)
        print("Created new wallet_balances.csv file.")

def load_balances():

    balances = {}
    with open(WALLET_FILE, mode='r', newline='') as file:
        reader = csv.reader(file)
        next(reader)
        for row in reader:

            currency = row[0]
            amount = float(row[1])
            balances[currency] = amount
    return balances

def save_balances(balances):

    with open(WALLET_FILE, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Currency', 'Balance'])
        for currency, amount in balances.items():
            writer.writerow([currency, round(amount, 2)])


def deposit_funds(balances):

    print("\n--- Deposit Funds ---")
    print("Available Currencies:", ", ".join(EXCHANGE_RATES.keys()))
    currency = input("Enter currency to deposit: ").upper()

    if currency not in EXCHANGE_RATES:
        print("Error: Unsupported currency.")
        return balances

    try:
        amount_input = input(f"Enter amount of {currency} to deposit: ")
        amount = float(amount_input)


        if amount <= 0:
            print("Error: Deposit amount must be greater than zero.")
            return balances

        balances[currency] += amount
        print(f"Success! Deposited {amount:.2f} {currency}.")

    except ValueError:

        print("Error: Invalid input. Please enter a numerical value.")

    return balances

def convert_currency(balances):

    print("\n--- Convert Currency ---")
    from_curr = input("Convert FROM (e.g., USD): ").upper()
    to_curr = input("Convert TO (e.g., EUR): ").upper()

    if from_curr not in EXCHANGE_RATES or to_curr not in EXCHANGE_RATES:
        print("Error: Unsupported currency selection.")
        return balances

    try:
        amount = float(input(f"Amount of {from_curr} to convert: "))

        if amount <= 0:
            print("Error: Amount must be greater than zero.")
            return balances

        if amount > balances[from_curr]:
            print(f"Error: Insufficient funds. You only have {balances[from_curr]:.2f} {from_curr}.")
            return balances


        amount_in_usd = amount / EXCHANGE_RATES[from_curr]
        final_amount = amount_in_usd * EXCHANGE_RATES[to_curr]


        balances[from_curr] -= amount
        balances[to_curr] += final_amount
        print(f"Success! Converted {amount:.2f} {from_curr} to {final_amount:.2f} {to_curr}.")

    except ValueError:
        print("Error: Invalid input. Please enter a numerical value.")

    return balances

def visualize_portfolio():

    print("\nGenerating Portfolio Visualization...")


    try:
        df = pd.read_csv(WALLET_FILE)
    except Exception as e:
        print(f"Error loading CSV for visualization: {e}")
        return


    def calculate_usd_value(row):
        return row['Balance'] / EXCHANGE_RATES[row['Currency']]

    df['USD_Value'] = df.apply(calculate_usd_value, axis=1)


    df_active = df[df['USD_Value'] > 0]

    if df_active.empty:
        print("Your wallet is empty! Deposit funds to see the visualization.")
        return


    plt.figure(figsize=(8, 6))
    plt.pie(df_active['USD_Value'], labels=df_active['Currency'], autopct='%1.1f%%', startangle=140, colors=['#4CAF50', '#2196F3', '#FFC107', '#E91E63'])
    plt.title('Wallet Net Worth Distribution (USD Equivalent)')
    plt.show()

def main():
    initialize_wallet()

    while True:
        balances = load_balances()

        print("\n=== MULTI-CURRENCY WALLET MANAGER ===")
        print("1. View Balances")
        print("2. Deposit Funds")
        print("3. Convert Currency")
        print("4. View Portfolio Chart")
        print("5. Ask AI Advisor") # <--- Add this line
        print("6. Exit")

        choice = input("Select an option (1-5): ")

        if choice == '1':
            print("\n--- Current Balances ---")
            for curr, amt in balances.items():
                print(f"{curr}: {amt:.2f}")
        elif choice == '2':
            updated_balances = deposit_funds(balances)
            save_balances(updated_balances)
        elif choice == '3':
            updated_balances = convert_currency(balances)
            save_balances(updated_balances)
        elif choice == '4':
            visualize_portfolio()

        elif choice == '5':
            consult_ai_advisor(balances) # <--- Add this new block
        elif choice == '6':
            print("Exiting application. Goodbye!")
            break
        else:
            print("Invalid choice. Please select a number from 1 to 5.")

if __name__ == "__main__":
    main()


=== MULTI-CURRENCY WALLET MANAGER ===
1. View Balances
2. Deposit Funds
3. Convert Currency
4. View Portfolio Chart
5. Ask AI Advisor
6. Exit

--- Current Balances ---
USD: 3000.00
EUR: 1000.00
GBP: 2000.79
JPY: 850.00

=== MULTI-CURRENCY WALLET MANAGER ===
1. View Balances
2. Deposit Funds
3. Convert Currency
4. View Portfolio Chart
5. Ask AI Advisor
6. Exit

--- Consulting AI Financial Advisor ---

Advisor is thinking...
ADVISOR RESPONSE: I can't give you an exact real-time exchange rate for USD to JPY.  

Here's why:

* **Currency rates fluctuate constantly.** The value of currencies changes based on market conditions and demand. Any number I give you now could be outdated within minutes! 
* **Exchange rates are determined by specific providers:** Companies like banks, financial institutions, and online platforms (like XE) handle currency exchange. They have their own algorithms and pricing structures.

**How to find the current exchange rate:**

1. **Use a reliable online source:*

In [2]:
# 1. Install zstd (The missing piece)
!apt-get update && apt-get install zstd -y

# 2. Re-run the Ollama installation
!curl -L https://ollama.com/install.sh | sh

# 3. Start the server brain in the background
import subprocess
import time

print("Starting Ollama server...")
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# 4. Pull the model
!ollama pull gemma2:2b
print("\n--- Success! The brain is now ready for your commands, My Sultan! ---")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,985 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.1 MB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,863 kB]
Get:14 http:/